In [ ]:
%matplotlib inline
import datetime
import pathlib as pl

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import IntSlider, interact
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

## Load the existing model

In [ ]:
sample_frequency = "monthly"  # monthly or annual

In [ ]:
# load the simple model setup

name = "sv"
ws = pl.Path("../data/synthetic-valley/synthetic-valley-vg")
new_ws = pl.Path(f"models/{ws.name}")
gwt_ws = new_ws.parent / "synthetic-valley-vg-gwt"

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
gwf = sim.get_model(name)
nlay, ncpl = gwf.disv.nlay.array, gwf.disv.ncpl.array
shape3d = (nlay, ncpl)

nper = sim.tdis.nper.array
perlen = sim.tdis.perioddata.array["perlen"]

# Plot gwf model data

In [ ]:
# shared settings for the gwf map figures
gwf_figsize = (17.15 / 2.541, 1.4 * 0.8333 * 17.15 / 2.541)
gwf_mosaic = [[0, 1, 2], [3, 4, "."]]

idomain = gwf.disv.idomain.array

# horizontal hydraulic conductivity by layer
k11 = np.ma.masked_where(idomain < 1, gwf.npf.k.array)
kpos = k11.compressed()
norm_k = mpl.colors.LogNorm(vmin=kpos[kpos > 0].min(), vmax=kpos.max())

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        gwf_mosaic,
        figsize=gwf_figsize,
        layout="constrained",
    )
    fig.suptitle("Horizontal hydraulic conductivity (m/d)")

    for k in range(nlay):
        ax = axd[k]
        ax.set_xlim(gwf.modelgrid.extent[:2])
        ax.set_ylim(gwf.modelgrid.extent[2:])
        ax.set_title(f"Layer {k + 1}")
        mm = flopy.plot.PlotMapView(
            model=gwf,
            ax=ax,
            extent=gwf.modelgrid.extent,
            layer=k,
        )
        mm.plot_ibound(color_vpt="blue", alpha=0.25)
        pa = mm.plot_array(k11, norm=norm_k, edgecolor="black", lw=0.25)
        mm.plot_bc(package=gwf.sfr, color="cyan")
        mm.plot_bc(package=gwf.wel, color="red")

    fig.colorbar(
        pa,
        ax=[axd[k] for k in range(nlay)],
        shrink=0.6,
        label="K (m/d)",
    )

In [ ]:
# cell bottom elevation by layer
botm = np.ma.masked_where(idomain < 1, gwf.disv.botm.array)

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        gwf_mosaic,
        figsize=gwf_figsize,
        layout="constrained",
    )
    fig.suptitle("Cell bottom elevation (m)")

    for k in range(nlay):
        ax = axd[k]
        ax.set_xlim(gwf.modelgrid.extent[:2])
        ax.set_ylim(gwf.modelgrid.extent[2:])
        ax.set_title(f"Layer {k + 1}")
        mm = flopy.plot.PlotMapView(
            model=gwf,
            ax=ax,
            extent=gwf.modelgrid.extent,
            layer=k,
        )
        mm.plot_ibound(color_vpt="blue", alpha=0.25)
        pa = mm.plot_array(
            botm,
            vmin=botm.min(),
            vmax=botm.max(),
            cmap="terrain",
            edgecolor="black",
            lw=0.25,
        )
        mm.plot_bc(package=gwf.sfr, color="cyan")

    fig.colorbar(
        pa,
        ax=[axd[k] for k in range(nlay)],
        shrink=0.6,
        label="Elevation (m)",
    )

In [ ]:
# single figure of the model top
extent = gwf.modelgrid.extent
xext = extent[1] - extent[0]
yext = extent[3] - extent[2]
w = 17.15 / 2.541 / 2
single_figsize = (w, w * yext / xext)

top = np.ma.masked_where(idomain[0] < 1, gwf.disv.top.array)

# center-column cross-section location (matches the cross-section figure)
xc = 0.5 * (extent[0] + extent[1])

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=single_figsize, layout="constrained")
    ax.set_xlim(extent[:2])
    ax.set_ylim(extent[2:])
    ax.set_title("Model top (m)")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, extent=extent, layer=0)
    mm.plot_ibound(color_vpt="blue", alpha=0.25)
    pa = mm.plot_array(top, cmap="terrain", edgecolor="black", lw=0.25)
    mm.plot_bc(package=gwf.sfr, color="cyan")

    # show the cross-section line
    ax.plot([xc, xc], extent[2:], color="red", lw=1.5, ls="--")
    ax.text(
        xc, extent[3], "A", color="red", fontweight="bold", ha="center", va="bottom"
    )
    ax.text(xc, extent[2], "A'", color="red", fontweight="bold", ha="center", va="top")

    fig.colorbar(pa, ax=ax, shrink=0.8, label="Elevation (m)")

In [ ]:
# cross-section along section A-A' (center column) showing horizontal K
extent = gwf.modelgrid.extent
xc = 0.5 * (extent[0] + extent[1])
# order the line A (north/top) -> A' (south/bottom) to match the map labels
line = {"line": [(xc, extent[3]), (xc, extent[2])]}

w = 17.15 / 2.541
with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(w, 0.5 * w), layout="constrained")
    ax.set_title("Horizontal hydraulic conductivity along section A-A'")
    xs = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line=line)
    pa = xs.plot_array(gwf.npf.k.array, norm=norm_k)
    xs.plot_grid(lw=0.3, color="0.5")
    xs.plot_ibound()
    ax.set_xlabel("Distance along section (m)")
    ax.set_ylabel("Elevation (m)")

    # label the section endpoints to match the model-top figure
    ax.text(
        0.0,
        1.01,
        "A",
        color="red",
        fontweight="bold",
        ha="left",
        va="bottom",
        transform=ax.transAxes,
    )
    ax.text(
        1.0,
        1.01,
        "A'",
        color="red",
        fontweight="bold",
        ha="right",
        va="bottom",
        transform=ax.transAxes,
    )

    fig.colorbar(pa, ax=ax, shrink=0.8, label="K (m/d)")

In [ ]:
# recharge rate applied to non-lake cells, by stress period
# (the rcha array is spatially uniform within each stress period)
perlen = sim.tdis.perioddata.array["perlen"]
rch_arr = gwf.rcha.recharge.array  # (nper, 1, ncpl)
rch_rate = rch_arr.reshape(perlen.size, -1).mean(axis=1)  # m/d

# convert the daily rate (m/d) to a monthly depth (mm/month) using each
# stress period's length (days) and 1000 mm/m
rch_month = rch_rate * perlen * 1000.0

# period boundaries (days) relative to START_DATE_TIME = 1962-01-01
start_date = datetime.datetime(1962, 1, 1)
bounds = np.concatenate(([0.0], np.cumsum(perlen)))  # len nper + 1

# exclude the steady-state spin-up (period 1); keep the monthly transient
monthly = np.arange(1, perlen.size)
x_days = np.append(bounds[monthly], bounds[-1])  # period starts + final end time
y = np.append(rch_month[monthly], rch_month[monthly][-1])
dates = [start_date + datetime.timedelta(days=float(d)) for d in x_days]

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(
        figsize=(17.15 / 2.541, 0.45 * 17.15 / 2.541),
        layout="constrained",
    )
    ax.plot(dates, y, drawstyle="steps-post", color="blue", lw=1.0)
    ax.fill_between(dates, y, step="post", color="blue", alpha=0.2)
    ax.set_xlim(dates[0], dates[-1])
    ax.set_ylim(bottom=0.0)
    flopy.plot.styles.heading(
        ax=ax, heading="Applied net recharge during the transient simulation period"
    )
    flopy.plot.styles.xlabel(ax=ax, label="Date")
    flopy.plot.styles.ylabel(ax=ax, label="Recharge rate (mm/month)")

## Run the gwf model

In [ ]:
sim.run_simulation()

# Plot the simulated gwf results

In [ ]:


# simulated heads through time (six-panel map; one frame per output time)
head_times = gwf.output.head().get_times()

extent = gwf.modelgrid.extent
xc = 0.5 * (extent[0] + extent[1])  # center-column cross-section (A-A')

head_mosaic = [[0, 1, 2], [3, 4, "cbar"]]
head_vmin, head_vmax = -2, 4
head_levels = np.arange(-2, 5, 1)
head_ticks = list(range(-2, 5))


def plot_head(time_index):
    totim = head_times[time_index]
    head = gwf.output.head().get_data(totim=totim)

    with flopy.plot.styles.USGSMap():
        fig, axd = plt.subplot_mosaic(
            head_mosaic,
            figsize=gwf_figsize,
            layout="constrained",
        )
        fig.suptitle(f"Simulated head (m), time = {totim:g} days (index {time_index})")

        for k in range(nlay):
            ax = axd[k]
            ax.set_xlim(extent[:2])
            ax.set_ylim(extent[2:])
            ax.set_title(f"Layer {k + 1}")
            mm = flopy.plot.PlotMapView(
                model=gwf,
                ax=ax,
                extent=extent,
                layer=k,
            )
            mm.plot_ibound(color_vpt="blue", alpha=0.25)
            pa = mm.plot_array(
                head,
                vmin=head_vmin,
                vmax=head_vmax,
                edgecolor="black",
                lw=0.25,
                masked_values=[1e30],
            )
            mm.plot_bc(package=gwf.sfr, color="cyan")
            mm.plot_bc(package=gwf.wel, color="red")

            cs = mm.contour_array(
                head,
                levels=head_levels,
                colors="black",
                linewidths=0.5,
                masked_values=[1e30],
            )
            ax.clabel(
                cs,
                inline=True,
                fmt="%g",
                fontsize=8,
                inline_spacing=0.5,
            )

            # show the cross-section location on the Layer 1 panel
            if k == 0:
                ax.plot([xc, xc], extent[2:], color="red", lw=1.5, ls="--")
                ax.text(
                    xc, extent[3], "A", color="red", fontweight="bold",
                    ha="center", va="bottom",
                )
                ax.text(
                    xc, extent[2], "A'", color="red", fontweight="bold",
                    ha="center", va="top",
                )

        # whole-number colorbar in the empty mosaic panel
        axd["cbar"].axis("off")
        cax = inset_axes(axd["cbar"], width="25%", height="85%", loc="center")
        cb = fig.colorbar(pa, cax=cax, ticks=head_ticks)
        cb.ax.yaxis.set_major_formatter(
            mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}")
        )
        cb.set_label("Head (m)")

        plt.show()


interact(
    plot_head,
    time_index=IntSlider(
        min=0,
        max=len(head_times) - 1,
        step=1,
        value=len(head_times) - 1,
        description="time index",
        continuous_update=False,
    ),
);

In [ ]:
# simulated heads along section A-A' (center column) through time
line = {"line": [(xc, extent[3]), (xc, extent[2])]}  # A (top) -> A' (bottom)
xs_w = 17.15 / 2.541


def plot_head_xs(time_index):
    totim = head_times[time_index]
    head = gwf.output.head().get_data(totim=totim)

    with flopy.plot.styles.USGSMap():
        fig, ax = plt.subplots(figsize=(xs_w, 0.5 * xs_w), layout="constrained")
        ax.set_title(
            f"Simulated head along A-A', time = {totim:g} days (index {time_index})"
        )
        xs = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line=line)
        pa = xs.plot_array(
            head, head=head, vmin=head_vmin, vmax=head_vmax, masked_values=[1e30],
        )
        xs.plot_grid(lw=0.3, color="0.5")
        xs.plot_ibound()

        cs = xs.contour_array(
            head,
            head=head,
            levels=head_levels,
            colors="black",
            linewidths=0.5,
            masked_values=[1e30],
        )
        ax.clabel(cs, inline=True, fmt="%g", fontsize=8, inline_spacing=0.5)

        ax.set_xlabel("Distance along section (m)")
        ax.set_ylabel("Elevation (m)")

        # label the section endpoints to match the map figure
        ax.text(
            0.0, 1.01, "A", color="red", fontweight="bold",
            ha="left", va="bottom", transform=ax.transAxes,
        )
        ax.text(
            1.0, 1.01, "A'", color="red", fontweight="bold",
            ha="right", va="bottom", transform=ax.transAxes,
        )

        cb = fig.colorbar(pa, ax=ax, shrink=0.8, ticks=head_ticks)
        cb.ax.yaxis.set_major_formatter(
            mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}")
        )
        cb.set_label("Head (m)")

        plt.show()


interact(
    plot_head_xs,
    time_index=IntSlider(
        min=0,
        max=len(head_times) - 1,
        step=1,
        value=len(head_times) - 1,
        description="time index",
        continuous_update=False,
    ),
);

### Create the groundwater transport model

### Create the groundwater transport model

In [ ]:
gwt_name = f"{name}_gwt"

In [ ]:
sim_gwt = flopy.mf6.MFSimulation(
    sim_name=gwt_name,
    sim_ws=gwt_ws,
    # continue_=True,
)

perioddata = sim.tdis.perioddata.array
# perioddata["nstp"][1:] = 20
# perioddata["tsmult"][1:] = 1.0
tdis = flopy.mf6.ModflowTdis(
    sim_gwt,
    time_units="DAYS",
    nper=nper,
    perioddata=perioddata,
)
ims = flopy.mf6.ModflowIms(
    sim_gwt,
    filename=f"{gwt_name}.ims",
    print_option="all",
    complexity="simple",
    outer_maximum=500,
    inner_maximum=200,
    linear_acceleration="bicgstab",
)


In [ ]:
gwt = flopy.mf6.ModflowGwt(
    sim_gwt,
    modelname=gwt_name,
    # dependent_variable_scaling=True,
)
dis = flopy.mf6.ModflowGwtdisv(
    gwt,
    length_units="meters",
    nlay=nlay,
    ncpl=ncpl,
    cell2d=gwf.disv.cell2d.array,
    nvert=gwf.disv.nvert.array,
    vertices=gwf.disv.vertices.array,
    top=gwf.disv.top.array,
    botm=gwf.disv.botm.array,
    idomain=gwf.disv.idomain.array,
)
ic = flopy.mf6.ModflowGwtic(gwt, strt=0.0)
adv = flopy.mf6.ModflowGwtadv(
    gwt,
    # ats_percel=0.75,
    scheme="utvd",
    # scheme="upstream",
)
dsp = flopy.mf6.ModflowGwtdsp(gwt, diffc=0.0e-12, alh=75.0, ath1=7.5)
mst = flopy.mf6.ModflowGwtmst(
    gwt,
    porosity=gwf.sto.sy.array,
    # zero_order_decay=True,
    # decay=-1.0 / 365.25,
)
pd = [
    ("GWFHEAD", f"../synthetic-valley-vg/{name}.hds", None),
    ("GWFBUDGET", f"../synthetic-valley-vg/{name}.cbc", None),
]
fmi = flopy.mf6.ModflowGwtfmi(gwt, packagedata=pd)
sourcerecarray = [
    ("LAK-1", "AUX", "CONCENTRATION"),
]
ssm = flopy.mf6.ModflowGwtssm(gwt, sources=sourcerecarray)
oc = flopy.mf6.ModflowGwtoc(
    gwt,
    concentration_filerecord=f"{name}.ucn",
    saverecord={
        1: [
            ("CONCENTRATION", "LAST"),
        ]
    },
    printrecord=[("BUDGET", "ALL")],
)

# add mover package
# mvt = flopy.mf6.ModflowGwtmvt(gwt)


In [ ]:
sim_gwt.write_simulation()

In [ ]:
sim_gwt.run_simulation()

## Plot the results

In [ ]:
times = gwt.output.concentration().get_times()
conc = gwt.output.concentration().get_data(totim=times[-1])
conc[conc < 1e30].min(), conc[conc < 1e30].max()

In [ ]:
c = gwt.output.concentration().get_data(totim=times[0])

In [ ]:
six_panel_figsize = (17.15 / 2.541, 1.4 * 0.8333 * 17.15 / 2.541)
mosaic = [[0, 1, 2], [3, 4, "."]]

In [ ]:
contour_color = "white"
contour_style = "--"

sv_gwt_contour_dict = {
    "linewidths": 0.75,
    "colors": contour_color,
    "linestyles": contour_style,
}

In [ ]:
times = gwt.output.concentration().get_times()

# reuse the six-panel layout but turn the empty slot into a colorbar axes
conc_mosaic = [[0, 1, 2], [3, 4, "cbar"]]


def plot_concentration(time_index):
    totim = times[time_index]
    conc = gwt.output.concentration().get_data(totim=totim)

    with flopy.plot.styles.USGSMap():
        fig, axd = plt.subplot_mosaic(
            conc_mosaic,
            figsize=six_panel_figsize,
            layout="constrained",
        )
        fig.suptitle(f"Time = {totim:g} days (time index {time_index})")

        norm = mpl.colors.LogNorm(vmin=0.01, vmax=100)
        for k in range(nlay):
            ax = axd[k]
            ax.set_xlim(gwf.modelgrid.extent[:2])
            ax.set_ylim(gwf.modelgrid.extent[2:])
            ax.set_title(f"Layer {k + 1}")
            mm = flopy.plot.PlotMapView(
                model=gwt,
                ax=ax,
                extent=gwt.modelgrid.extent,
                layer=k,
            )
            mm.plot_ibound(color_vpt="blue", alpha=0.25)
            pa = mm.plot_array(
                conc,
                edgecolor="black",
                lw=0.25,
                masked_values=[0],
                norm=norm,
            )

            mm.plot_bc(package=gwf.sfr, color="cyan")
            mm.plot_bc(package=gwf.wel, color="red")

            c = conc[k]
            cmax = c[c < 1e30].max()
            contour_array = cmax >= 0.1
            if contour_array:
                cs = mm.contour_array(
                    conc,
                    # masked_values=[0],
                    levels=[0.01, 1, 10, 100],
                    **sv_gwt_contour_dict,
                )

                ax.clabel(
                    cs,
                    inline=True,
                    fmt="%g",
                    fontsize=8,
                    inline_spacing=0.5,
                    colors="white",
                )

        # slim vertical colorbar centered in the empty mosaic panel
        axd["cbar"].axis("off")
        cax = inset_axes(
            axd["cbar"],
            width="25%",
            height="85%",
            loc="center",
        )
        ticks = [0.01, 0.1, 1, 10, 100]
        cb = fig.colorbar(pa, cax=cax, ticks=ticks)
        cb.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}"))
        cb.minorticks_off()
        cb.set_label("Concentration")

        plt.show()


interact(
    plot_concentration,
    time_index=IntSlider(
        min=0,
        max=len(times) - 1,
        step=1,
        value=len(times) - 1,
        description="time index",
        continuous_update=False,
    ),
);